In [2]:
import torch
import os
from PIL import Image
import clip
import os.path as osp
import os, sys
import torch
import numpy as np
import pandas as pd
import ast
from transformers import CLIPTokenizer
import math
import json
from torch.utils.data import Dataset, DataLoader, default_collate
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import requests
from torchvision import transforms
import torch.nn.functional as F
from transformers import Owlv2Processor, Owlv2ForObjectDetection
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.nn as nn
import seaborn as sns
import torchvision.utils as vutils
sys.path.insert(0, '../')

from lib.utils import load_model_weights,mkdir_p
from models.GALIP import NetG, CLIP_TXT_ENCODER

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

## Generating Images

In [4]:
CLIP_text = "ViT-B/32"
clip_model, preprocess = clip.load("ViT-B/32")
clip_model.load_state_dict(torch.load("../../models/NegCLIP_CC12M_NegFull/checkpoint.pt", weights_only=False)['state_dict'])
clip_model = clip_model.to(device)
clip_model = clip_model.eval()

for param in clip_model.parameters():
    param.requires_grad = False



text_encoder = CLIP_TXT_ENCODER(clip_model).to(device)
netG = NetG(64, 100, 512, 256, 3, False, clip_model).to(device)
path = '../saved_models/pretrained/pre_cc12m.pth'
checkpoint = torch.load(path, map_location=torch.device('cpu'))
netG = load_model_weights(netG, checkpoint['model']['netG'], multi_gpus=False)

In [ ]:
os.makedirs("samples/neg-clip-negfull", exist_ok=True)
os.makedirs("samples/neg-clip-negfull+ours", exist_ok=True)

In [ ]:
batch_size = 8

selected_captions = pd.read_json("prompts_with_questions.json")['Prompt'].values
splitted_caps = pd.read_csv("splitted_prompts.csv", index_col="Prompts")

for i, caption in tqdm(enumerate(selected_captions), total=len(selected_captions)):
    noise = torch.randn((batch_size, 100)).to(device)
    # generate from text
    with torch.no_grad():
                    
        batch_aff_texts = [splitted_caps.loc[caption, "Affirmative"]]
        batch_not_texts = [splitted_caps.loc[caption, "Negated"]]

        batch_aff_texts_tok = clip.tokenize(batch_aff_texts, truncate=True).to(device)
        batch_not_texts_tok = clip.tokenize(batch_not_texts, truncate=True).to(device)

        batch_aff_texts_emb = F.normalize(clip_model.encode_text(batch_aff_texts_tok), dim=-1) # a : B x d
        batch_not_texts_emb = F.normalize(clip_model.encode_text(batch_not_texts_tok), dim=-1) # n : B x d
        alpha = math.acos(0.9) 
        a_T_n = torch.sum(batch_aff_texts_emb * batch_not_texts_emb, dim=-1) # cos(theta) : B,
        theta = torch.acos(a_T_n) # Theta : B,
        mask = theta < 2*alpha

        delta = alpha + theta/2.
        sent_emb = torch.cos(delta)[:, None] * batch_not_texts_emb + (torch.sin(delta)/torch.sin(theta))[:, None] * (batch_aff_texts_emb - a_T_n[:, None]*batch_not_texts_emb)        
        tokenized_text = clip.tokenize([caption]).to(device)
        ref_emb, _ = text_encoder(tokenized_text)
        sent_emb = sent_emb *  ref_emb.norm(keepdim=True)
        sent_emb = sent_emb.repeat(batch_size,1)
        ref_emb = ref_emb.repeat(batch_size, 1)

        ref_fake_imgs = netG(noise,ref_emb,eval=True).float()
        fake_imgs = netG(noise,sent_emb,eval=True).float()
        
        vutils.save_image(ref_fake_imgs.data, f'./samples/neg-clip-negfull/{caption.replace(" ","-")}.png', nrow=8, value_range=(-1, 1), normalize=True)
        vutils.save_image(fake_imgs.data, f'./samples/neg-clip-negfull+ours/{caption.replace(" ","-")}.png', nrow=8, value_range=(-1, 1), normalize=True)

  0%|          | 0/107 [00:00<?, ?it/s]

100%|██████████| 107/107 [00:41<00:00,  2.56it/s]


## Evalution of Genrated Images

In [ ]:
from transformers import pipeline
import torch

pipe = pipeline(
    "image-text-to-text",
    model="google/gemma-3-27b-it",
    device_map="auto",
    torch_dtype=torch.bfloat16
)

In [ ]:
with torch.no_grad():
    imgs_name = os.listdir("samples/neg-clip-negfull+ours")

    all_aff_scores = []
    all_neg_scores = []
    all_scores = []

    df = pd.read_json("prompts_with_questions.json").set_index("Prompt")

    for img_name in tqdm(imgs_name, total=len(imgs_name)):

        caption = img_name.split('.png')[0]
        caption = caption.replace('-', ' ')
        
        ref_img = vutils.Image.open(os.path.join("samples/neg-clip-negfull", img_name))
        
        w, h = ref_img.size
        num_images = 8
        img_width = w // num_images
        img_height = h  # single row

        # Split and save each image
        aff_scores = []
        neg_scores = []
        scores = []
        for i in range(num_images):
            left = i * img_width
            right = (i + 1) * img_width
            img = ref_img.crop((left, 0, right, img_height))

            messages = [
                {
                    "role": "system",
                    "content": [{"type": "text", "text": "You are a ahelpful assistant. Answer the question based on the image with 'Yes' or 'No' , without use of any addtional characters or words."}]
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img},
                        {"type": "text", "text": df.loc[caption, "Question1"]}
                    ]
                }
            ]
            output = pipe(text=messages, max_new_tokens=10)[0]["generated_text"][-1]["content"].strip().lower()
            
            if output not in ["yes", "no", "yes.", "no."]:
                print(f"Unexpected output1: {output}")

            if output == "yes" or output == "yes.":
                aff_scores.append(1.)
            else:
                aff_scores.append(0.)
            


            messages = [
                {
                    "role": "system",
                    "content": [{"type": "text", "text": "You are a ahelpful assistant. Answer the question based on the image with 'Yes' or 'No'."}]
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img},
                        {"type": "text", "text": df.loc[caption, "Question2"]}
                    ]
                }
            ]
            output = pipe(text=messages, max_new_tokens=10)[0]["generated_text"][-1]["content"].strip().lower()
            
            if output not in ["yes", "no", "yes.", "no."]:
                print(f"Unexpected output2: {output}")
            
            if output == "no" or output == "no.":
                neg_scores.append(1.)
            else:
                neg_scores.append(0.)


            scores.append(aff_scores[-1] * neg_scores[-1])

        all_aff_scores.append(np.mean(aff_scores).item())
        all_neg_scores.append(np.mean(neg_scores).item())
        all_scores.append(np.mean(scores).item())

        print("aff scores: ", np.mean(all_aff_scores))
        print("neg scores: ", np.mean(all_neg_scores))
        print("scores: ", np.mean(all_scores))